In [1]:
import os
import math
import json
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
import matplotlib.colors
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from datetime import datetime
# from patsy import dmatrices
from patsy import dmatrix
from scipy.stats import ttest_rel
from scipy.stats import ttest_ind
import statsmodels.formula.api as smf
main_path = r'/home/20250114zmz_kd/'

In [2]:
from causalinference import CausalModel
from causalinference.utils import random_data

In [3]:
CIs = {'90': 1.645, '95': 1.96, '99': 2.576}

In [4]:
labels = ['US', 'Others']

In [5]:
data = r'GraduationPaper/RevisetoJournal/9991-MergedData_similarity.csv'
d = pd.read_csv(main_path + data)
del d['Unnamed: 0']
print(d .shape)
d .columns

(317275, 102)


Index(['work_id', 'PublishedYear', 'Facility', 'num_fac', 'paper_type',
       'paper_language', 'novel_uzzi', 'novel_uzzi_bin', 'num_fac_scientist',
       'ratio_fac_scientist',
       ...
       'ab_length', 'mean_career_age', 'ex_ld_avg_avgimpact',
       'ex_ld_avg_insthindex', 'ex_ld_avg_before_year_prod_fac',
       'ex_ld_avg_before_year_with_ih', 'ex_ld_avg_before_year_co_lead',
       'ex_ld_avg_before_year_participation', 'knowledge_proximity_mean',
       'knowledge_proximity_max'],
      dtype='object', length=102)

In [6]:
d['text_fac_scientist'].unique()

array(['StaffPart', 'NonStaffPart'], dtype=object)

In [7]:
d = d[d['text_fac_scientist'].isin(['StaffPart','NonStaffPart'])]
print(d .shape)

(317275, 102)


In [8]:
d['reg_class_bin'] = d.apply(lambda row: 1 if row['text_fac_scientist'] == 'StaffPart' else 0, axis = 1)
d['reg_class_bin'].value_counts()

reg_class_bin
0    232552
1     84723
Name: count, dtype: int64

In [9]:
# 假设 df 是你的 dataframe
d['PublishedYear'] = d['PublishedYear'].astype('category')

In [10]:
d .columns

Index(['work_id', 'PublishedYear', 'Facility', 'num_fac', 'paper_type',
       'paper_language', 'novel_uzzi', 'novel_uzzi_bin', 'num_fac_scientist',
       'ratio_fac_scientist',
       ...
       'mean_career_age', 'ex_ld_avg_avgimpact', 'ex_ld_avg_insthindex',
       'ex_ld_avg_before_year_prod_fac', 'ex_ld_avg_before_year_with_ih',
       'ex_ld_avg_before_year_co_lead', 'ex_ld_avg_before_year_participation',
       'knowledge_proximity_mean', 'knowledge_proximity_max', 'reg_class_bin'],
      dtype='object', length=103)

In [13]:
co_feats_ = ["lnnum_author", "international", "lnnum_reference", "num_fac", "SDG",
             "lnmean_career_age", "lnex_ld_avg_avgimpact", "lnex_ld_avg_insthindex",
             "ex_ld_bin_gs", "ex_ld_bin_sameC", "knowledge_proximity_mean",
             "lnex_ld_avg_before_year_prod_fac", "ex_ld_max_before_year_with_ih_bin",
             'Agricultural and Biological Sciences',
             'Arts and Humanities', 'Biochemistry, Genetics and Molecular Biology',
             'Business, Management and Accounting', 'Chemical Engineering',
             'Chemistry', 'Computer Science', 'Decision Sciences', 'Dentistry',
             'Earth and Planetary Sciences', 'Economics, Econometrics and Finance',
             'Energy', 'Engineering', 'Environmental Science', 'Health Professions',
             'Immunology and Microbiology', 'Materials Science', 'Mathematics',
             'Medicine', 'Neuroscience', 'Nursing','Pharmacology, Toxicology and Pharmaceutics', 'Physics and Astronomy',
             'Psychology', 'Social Sciences', 'Veterinary',]

In [14]:
# 1. 把 DataFrame 里的空格替换成下划线
d.columns = d.columns.str.replace(' ', '_')
d.columns = d.columns.str.replace(',', '')
print(d .columns)
# 2. 把特征列表里的空格也替换掉
co_feats_ = [f.replace(' ', '_') for f in co_feats_]
co_feats_ = [f.replace(',', '') for f in co_feats_]
print(co_feats_)

Index(['work_id', 'PublishedYear', 'Facility', 'num_fac', 'paper_type',
       'paper_language', 'novel_uzzi', 'novel_uzzi_bin', 'num_fac_scientist',
       'ratio_fac_scientist',
       ...
       'mean_career_age', 'ex_ld_avg_avgimpact', 'ex_ld_avg_insthindex',
       'ex_ld_avg_before_year_prod_fac', 'ex_ld_avg_before_year_with_ih',
       'ex_ld_avg_before_year_co_lead', 'ex_ld_avg_before_year_participation',
       'knowledge_proximity_mean', 'knowledge_proximity_max', 'reg_class_bin'],
      dtype='object', length=103)
['lnnum_author', 'international', 'lnnum_reference', 'num_fac', 'SDG', 'lnmean_career_age', 'lnex_ld_avg_avgimpact', 'lnex_ld_avg_insthindex', 'ex_ld_bin_gs', 'ex_ld_bin_sameC', 'knowledge_proximity_mean', 'lnex_ld_avg_before_year_prod_fac', 'ex_ld_max_before_year_with_ih_bin', 'Agricultural_and_Biological_Sciences', 'Arts_and_Humanities', 'Biochemistry_Genetics_and_Molecular_Biology', 'Business_Management_and_Accounting', 'Chemical_Engineering', 'Chemistry', 'Comp

In [15]:
X = dmatrix(formula_like=' + '.join(co_feats_), data=d, return_type="dataframe")

In [16]:
list(X.columns)

['Intercept',
 'international[T.international]',
 'SDG[T.True]',
 'ex_ld_bin_gs[T.GlobalSouth]',
 'ex_ld_bin_sameC[T.Same]',
 'ex_ld_max_before_year_with_ih_bin[T.True]',
 'lnnum_author',
 'lnnum_reference',
 'num_fac',
 'lnmean_career_age',
 'lnex_ld_avg_avgimpact',
 'lnex_ld_avg_insthindex',
 'knowledge_proximity_mean',
 'lnex_ld_avg_before_year_prod_fac',
 'Agricultural_and_Biological_Sciences',
 'Arts_and_Humanities',
 'Biochemistry_Genetics_and_Molecular_Biology',
 'Business_Management_and_Accounting',
 'Chemical_Engineering',
 'Chemistry',
 'Computer_Science',
 'Decision_Sciences',
 'Dentistry',
 'Earth_and_Planetary_Sciences',
 'Economics_Econometrics_and_Finance',
 'Energy',
 'Engineering',
 'Environmental_Science',
 'Health_Professions',
 'Immunology_and_Microbiology',
 'Materials_Science',
 'Mathematics',
 'Medicine',
 'Neuroscience',
 'Nursing',
 'Pharmacology_Toxicology_and_Pharmaceutics',
 'Physics_and_Astronomy',
 'Psychology',
 'Social_Sciences',
 'Veterinary']

In [17]:
co_feats = [
 'international[T.international]',
 'SDG[T.True]',
 'ex_ld_bin_gs[T.GlobalSouth]',
 'ex_ld_bin_sameC[T.Same]',
 'ex_ld_max_before_year_with_ih_bin[T.True]',
 'lnnum_author',
 'lnnum_reference',
 'num_fac',
 'lnmean_career_age',
 'lnex_ld_avg_avgimpact',
 'lnex_ld_avg_insthindex',
 'knowledge_proximity_mean',
 'lnex_ld_avg_before_year_prod_fac',
 'Agricultural_and_Biological_Sciences',
 'Arts_and_Humanities',
 'Biochemistry_Genetics_and_Molecular_Biology',
 'Business_Management_and_Accounting',
 'Chemical_Engineering',
 'Chemistry',
 'Computer_Science',
 'Decision_Sciences',
 'Dentistry',
 'Earth_and_Planetary_Sciences',
 'Economics_Econometrics_and_Finance',
 'Energy',
 'Engineering',
 'Environmental_Science',
 'Health_Professions',
 'Immunology_and_Microbiology',
 'Materials_Science',
 'Mathematics',
 'Medicine',
 'Neuroscience',
 'Nursing',
 'Pharmacology_Toxicology_and_Pharmaceutics',
 'Physics_and_Astronomy',
 'Psychology',
 'Social_Sciences',
 'Veterinary'
]

In [18]:
X['text_fac_scientist'] = d['text_fac_scientist']
X['reg_class_bin'] = d['reg_class_bin']
X['novel_uzzi_bin'] = d['novel_uzzi_bin']

In [19]:
# Y is the outcome, D is treatment status, and X is the independent variable
causal = CausalModel(Y=X['novel_uzzi_bin'].values, D=X['reg_class_bin'].values, \
                     X=X[co_feats].values)

In [20]:
print(causal.summary_stats)


Summary Statistics

                    Controls (N_c=232552)       Treated (N_t=84723)             
       Variable         Mean         S.d.         Mean         S.d.     Raw-diff
--------------------------------------------------------------------------------
              Y        0.373        0.484        0.364        0.481       -0.009

                    Controls (N_c=232552)       Treated (N_t=84723)             
       Variable         Mean         S.d.         Mean         S.d.     Nor-diff
--------------------------------------------------------------------------------
             X0        0.439        0.496        0.754        0.431        0.679
             X1        0.486        0.500        0.446        0.497       -0.080
             X2        0.063        0.243        0.067        0.250        0.016
             X3        0.531        0.499        0.281        0.449       -0.527
             X4        0.681        0.466        0.897        0.304        0.548
      

In [21]:
causal.est_propensity()

In [22]:
# Propensity model results
print(causal.propensity)


Estimated Parameters of Propensity Score

                    Coef.       S.e.          z      P>|z|      [95% Conf. int.]
--------------------------------------------------------------------------------
     Intercept     -2.495      0.075    -33.287      0.000     -2.642     -2.348
            X0      0.760      0.011     70.691      0.000      0.739      0.781
            X1     -0.043      0.009     -4.692      0.000     -0.061     -0.025
            X2     -0.395      0.018    -21.677      0.000     -0.431     -0.360
            X3     -0.540      0.011    -51.075      0.000     -0.560     -0.519
            X4      1.288      0.015     86.844      0.000      1.259      1.317
            X5      0.514      0.008     62.525      0.000      0.498      0.530
            X6      0.035      0.009      3.970      0.000      0.018      0.052
            X7      0.608      0.007     84.188      0.000      0.594      0.622
            X8      0.356      0.015     23.359      0.000      0.

In [23]:
causal.propensity['fitted']

array([0.43132111, 0.30836371, 0.13416165, ..., 0.13902641, 0.13801113,
       0.57928826])

In [25]:
d['text_fac_scientist'].unique()

array(['StaffPart', 'NonStaffPart'], dtype=object)

In [26]:
for feat in co_feats:
    print(feat)
    af_avg = np.mean(X.loc[X['text_fac_scientist']=='StaffPart', feat])
    nam_avg = np.mean(X.loc[X['text_fac_scientist']=='NonStaffPart', feat])
    print('\tCollaboration:\t', af_avg)
    print('\tService:\t', nam_avg)
    print('\tDiff:\t', af_avg-nam_avg)
    print('\tT-test:\t', ttest_ind(X.loc[X['text_fac_scientist']=='StaffPart', feat], X.loc[X['text_fac_scientist']=='NonStaffPart', feat])[1])

international[T.international]
	Collaboration:	 0.7539865207794814
	Service:	 0.43872338229729263
	Diff:	 0.31526313848218873
	T-test:	 0.0
SDG[T.True]
	Collaboration:	 0.4455932863567154
	Service:	 0.485650521173759
	Diff:	 -0.0400572348170436
	T-test:	 6.0153739249768674e-89
ex_ld_bin_gs[T.GlobalSouth]
	Collaboration:	 0.06703020431287844
	Service:	 0.06303536413361313
	Diff:	 0.00399484017926531
	T-test:	 4.815761418896845e-05
ex_ld_bin_sameC[T.Same]
	Collaboration:	 0.2808682412095889
	Service:	 0.5312059238363893
	Diff:	 -0.2503376826268004
	T-test:	 0.0
ex_ld_max_before_year_with_ih_bin[T.True]
	Collaboration:	 0.8966868500879336
	Service:	 0.6808369740961161
	Diff:	 0.21584987599181749
	T-test:	 0.0
lnnum_author
	Collaboration:	 2.2904522758355905
	Service:	 1.9980434311474133
	Diff:	 0.2924088446881772
	T-test:	 0.0
lnnum_reference
	Collaboration:	 3.6559259725780926
	Service:	 3.6534719171420424
	Diff:	 0.002454055436050151
	T-test:	 0.2883786348241335
num_fac
	Collaboration:	

In [27]:
len(causal.propensity['fitted'])

317275

In [28]:
X['pscore'] = causal.propensity['fitted']

In [29]:
tem = X[['text_fac_scientist', 'pscore']].sort_values(by = ['pscore'])
tem['index'] = tem.index

In [30]:
tem = tem.values.tolist()

In [31]:
tem[-10:]

[['StaffPart', 0.9926184272948217, 205046],
 ['StaffPart', 0.9926880970186529, 205045],
 ['StaffPart', 0.9963201474907949, 280659],
 ['StaffPart', 0.9963403348672561, 280657],
 ['StaffPart', 0.9963941505943268, 280656],
 ['StaffPart', 0.9964349199234598, 280658],
 ['StaffPart', 0.9978600313396897, 280654],
 ['StaffPart', 0.9978632640069264, 280660],
 ['StaffPart', 0.9978657954923685, 280655],
 ['StaffPart', 0.997866922242244, 280653]]

In [32]:
leng = len(tem)

In [33]:
print(leng)

317275


In [34]:
pairs = {}
for i, elms in enumerate(tem):
    gen, score, ix = elms
    if gen == 'StaffPart':
        pre_nam_ix, nex_nam_ix = 0, 0
        j = i-1
        while j >= 0:
            if tem[j][0] != 'NonStaffPart':
                j -= 1
            else:
                break
        if j >= 0:
            pre_nam_ix = j
        n = i+1
        while n <= leng-1:
            if tem[n][0] != 'NonStaffPart':
                n += 1
            else:
                break
        if n <= leng-1:
            nex_nam_ix = n
        if abs(score - tem[pre_nam_ix][1]) <= abs(score - tem[nex_nam_ix][1]):
            pairs[ix] = tem[pre_nam_ix][2]
        else:
            pairs[ix] = tem[nex_nam_ix][2]

In [35]:
len(pairs)

84723

In [36]:
for feat in co_feats:
    print('\n')
    print('Feat:', feat, '\n')
    print('Before matching:\n')
    af_avg = np.mean(X.loc[X['text_fac_scientist']=='StaffPart', feat])
    nam_avg = np.mean(X.loc[X['text_fac_scientist']=='NonStaffPart', feat])
    print('\tStaffPart:\t', af_avg)
    print('\tNonStaffPart:\t', nam_avg)
    print('\tDiff:\t', af_avg-nam_avg)
    print('\tT-test:\t', ttest_ind(X.loc[X['text_fac_scientist']=='StaffPart', feat], X.loc[X['text_fac_scientist']=='NonStaffPart', feat])[1])

    print('\nAfter matching:\n')
    af_avg = np.mean(X.loc[pairs.keys(), feat])
    nam_avg = np.mean(X.loc[pairs.values(), feat])
    print('\tStaffPart:\t', af_avg)
    print('\tNonStaffPart:\t', nam_avg)
    print('\tDiff:\t', af_avg-nam_avg)
    print('\tT-test:\t', ttest_rel(X.loc[pairs.keys(), feat], X.loc[pairs.values(), feat])[1])



Feat: international[T.international] 

Before matching:

	StaffPart:	 0.7539865207794814
	NonStaffPart:	 0.43872338229729263
	Diff:	 0.31526313848218873
	T-test:	 0.0

After matching:

	StaffPart:	 0.7539865207794814
	NonStaffPart:	 0.7485216529159733
	Diff:	 0.005464867863508083
	T-test:	 0.0007158540607722867


Feat: SDG[T.True] 

Before matching:

	StaffPart:	 0.4455932863567154
	NonStaffPart:	 0.485650521173759
	Diff:	 -0.0400572348170436
	T-test:	 6.0153739249768674e-89

After matching:

	StaffPart:	 0.4455932863567154
	NonStaffPart:	 0.44794211725269406
	Diff:	 -0.0023488308959786486
	T-test:	 0.33004533004371395


Feat: ex_ld_bin_gs[T.GlobalSouth] 

Before matching:

	StaffPart:	 0.06703020431287844
	NonStaffPart:	 0.06303536413361313
	Diff:	 0.00399484017926531
	T-test:	 4.815761418896845e-05

After matching:

	StaffPart:	 0.06703020431287844
	NonStaffPart:	 0.06590890313138109
	Diff:	 0.00112130118149735
	T-test:	 0.3530714025049331


Feat: ex_ld_bin_sameC[T.Same] 

Before m

In [37]:
for feat in co_feats:
    af_avg = np.mean(X.loc[X['text_fac_scientist']=='StaffPart', feat])
    nam_avg = np.mean(X.loc[X['text_fac_scientist']=='NonStaffPart', feat])
    af_avg_ = np.mean(X.loc[pairs.keys(), feat])
    nam_avg_ = np.mean(X.loc[pairs.values(), feat])
    print(feat, ' & ', '{:6.2f}'.format(af_avg), ' & ', '{:6.2f}'.format(nam_avg), ' & ', '{:6.2f}'.format(af_avg_), ' & ', '{:6.2f}'.format(nam_avg_), ' \\\\ \hline')

international[T.international]  &    0.75  &    0.44  &    0.75  &    0.75  \\ \hline
SDG[T.True]  &    0.45  &    0.49  &    0.45  &    0.45  \\ \hline
ex_ld_bin_gs[T.GlobalSouth]  &    0.07  &    0.06  &    0.07  &    0.07  \\ \hline
ex_ld_bin_sameC[T.Same]  &    0.28  &    0.53  &    0.28  &    0.29  \\ \hline
ex_ld_max_before_year_with_ih_bin[T.True]  &    0.90  &    0.68  &    0.90  &    0.90  \\ \hline
lnnum_author  &    2.29  &    2.00  &    2.29  &    2.32  \\ \hline
lnnum_reference  &    3.66  &    3.65  &    3.66  &    3.65  \\ \hline
num_fac  &    1.54  &    1.24  &    1.54  &    1.48  \\ \hline
lnmean_career_age  &    3.13  &    3.09  &    3.13  &    3.13  \\ \hline
lnex_ld_avg_avgimpact  &    2.97  &    3.00  &    2.97  &    2.97  \\ \hline
lnex_ld_avg_insthindex  &    5.96  &    6.10  &    5.96  &    5.96  \\ \hline
knowledge_proximity_mean  &    0.75  &    0.76  &    0.75  &    0.75  \\ \hline
lnex_ld_avg_before_year_prod_fac  &    2.32  &    2.19  &    2.32  &    2.26  

In [38]:
# AS avg
np.mean(X.loc[pairs.keys(), 'novel_uzzi_bin'])

np.float64(0.36398616668437145)

In [39]:
# NAM avg
np.mean(X.loc[pairs.values(), 'novel_uzzi_bin'])

np.float64(0.344015202483387)

In [40]:
ttest_rel(X.loc[pairs.keys(), 'novel_uzzi_bin'].apply(lambda x: 1 if x == True else 0), \
          X.loc[pairs.values(), 'novel_uzzi_bin'].apply(lambda x: 1 if x == True else 0))

TtestResult(statistic=np.float64(8.631904561116958), pvalue=np.float64(6.136155277677364e-18), df=np.int64(84722))

In [41]:
y_treated = X.loc[list(pairs.keys()), 'novel_uzzi_bin'].values
y_control = X.loc[list(pairs.values()), 'novel_uzzi_bin'].values

n_boot = 1000  # bootstrap次数
att_boot = np.zeros(n_boot)
n = len(y_treated)

for i in range(n_boot):
    idx = np.random.randint(0, n, n)  # 随机抽样索引，有放回
    att_boot[i] = np.mean(y_treated[idx] - y_control[idx])

# ATT估计值
att = np.mean(y_treated - y_control)

# 95%置信区间
ci_lower = np.percentile(att_boot, 2.5)
ci_upper = np.percentile(att_boot, 97.5)

print("ATT:", att)
print("95% CI: [{:.4f}, {:.4f}]".format(ci_lower, ci_upper))

ATT: 0.019970964200984383
95% CI: [0.0152, 0.0244]


In [42]:
d['novel_uzzi_bin'] = d['novel_uzzi_bin'].astype('int32')

In [43]:
print(d .shape)

(317275, 103)


In [44]:
dpsm = d.loc[list(pairs.keys()) + list(pairs.values())].sample(frac=1)
print(dpsm .shape)

(169446, 103)


In [42]:
dpsm .columns

Index(['work_id', 'PublishedYear', 'Facility', 'num_fac', 'paper_type',
       'paper_language', 'novel_uzzi', 'novel_uzzi_bin', 'num_fac_scientist',
       'ratio_fac_scientist',
       ...
       'mean_career_age', 'ex_ld_avg_avgimpact', 'ex_ld_avg_insthindex',
       'ex_ld_avg_before_year_prod_fac', 'ex_ld_avg_before_year_with_ih',
       'ex_ld_avg_before_year_co_lead', 'ex_ld_avg_before_year_participation',
       'knowledge_proximity_mean', 'knowledge_proximity_max', 'reg_class_bin'],
      dtype='object', length=103)

In [43]:
dpsm['work_id'].nunique()

41165

In [52]:
Update

NameError: name 'Update' is not defined

In [44]:
dpsm .to_csv(main_path + r'science_media_coverage/260128Revision/PSM-FirstCorresponding/PSM-sample-260311-africa.csv')